# Transformer：从零实现《Attention Is All You Need》

本 Notebook 使用 PyTorch 手写原论文的 **Encoder–Decoder Transformer**，包含：

- 正弦/余弦位置编码（Sinusoidal Positional Encoding）
- 缩放点积注意力与多头注意力（Multi-Head Attention）
- 位置前馈网络（Position-wise FFN）
- 残差连接、Dropout 与 LayerNorm
- Encoder / Decoder 堆叠
- Padding Mask 与 Decoder Causal Mask
- 原论文的 Noam 学习率调度与 Label Smoothing

为便于在普通电脑上运行，演示模型默认采用较小参数；将 `N=6, D_MODEL=512, N_HEADS=8, D_FF=2048` 即可对齐论文基础模型的主要尺寸。

In [42]:
import math
import random

import torch
import torch.nn as nn
import torch.nn.functional as F

# 保证示例尽量可复现
SEED = 42
random.seed(SEED)
torch.manual_seed(SEED)

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("device:", DEVICE)

# 特殊符号：PAD=0, BOS=1, EOS=2；其余编号表示普通 token
PAD_IDX, BOS_IDX, EOS_IDX = 0, 1, 2
VOCAB_SIZE = 32

# 小型演示配置（论文 base：N=6, D_MODEL=512, N_HEADS=8, D_FF=2048）
N = 2
D_MODEL = 64
N_HEADS = 4
D_FF = 256
DROPOUT = 0.1
MAX_LEN = 64

device: cuda


In [43]:
class PositionalEncoding(nn.Module):
    """论文中的固定正弦/余弦位置编码。"""

    def __init__(self, d_model, max_len=5000, dropout=0.1):
        super().__init__()
        self.dropout = nn.Dropout(dropout)

        position = torch.arange(max_len, dtype=torch.float32).unsqueeze(1)
        div_term = torch.exp(
            torch.arange(0, d_model, 2, dtype=torch.float32)
            * (-math.log(10000.0) / d_model)
        )
        pe = torch.zeros(max_len, d_model)
        pe[:, 0::2] = torch.sin(position * div_term)
        pe[:, 1::2] = torch.cos(position * div_term)
        self.register_buffer("pe", pe.unsqueeze(0))  # (1, max_len, d_model)

    def forward(self, x):
        # x: (batch, seq_len, d_model)
        x = x + self.pe[:, :x.size(1)]
        return self.dropout(x)


class MultiHeadAttention(nn.Module):
    """MultiHead(Q, K, V) = Concat(head_1, ..., head_h) W_O。"""

    def __init__(self, d_model, n_heads, dropout=0.1):
        super().__init__()
        if d_model % n_heads != 0:
            raise ValueError("d_model 必须能被 n_heads 整除")

        self.n_heads = n_heads
        self.d_k = d_model // n_heads
        self.q_proj = nn.Linear(d_model, d_model)
        self.k_proj = nn.Linear(d_model, d_model)
        self.v_proj = nn.Linear(d_model, d_model)
        self.out_proj = nn.Linear(d_model, d_model)
        self.dropout = nn.Dropout(dropout)

    def _split_heads(self, x):
        batch_size, seq_len, _ = x.shape
        # (B, L, D) -> (B, H, L, D/H)
        return x.view(batch_size, seq_len, self.n_heads, self.d_k).transpose(1, 2)

    def forward(self, query, key, value, mask=None, return_attention=False):
        q = self._split_heads(self.q_proj(query))
        k = self._split_heads(self.k_proj(key))
        v = self._split_heads(self.v_proj(value))

        # Scaled Dot-Product Attention
        scores = torch.matmul(q, k.transpose(-2, -1)) / math.sqrt(self.d_k)
        if mask is not None:
            # mask=True 表示该位置可见；形状需可广播至 (B, H, Lq, Lk)
            scores = scores.masked_fill(~mask, torch.finfo(scores.dtype).min)

        attention = self.dropout(F.softmax(scores, dim=-1))
        context = torch.matmul(attention, v)

        batch_size, _, query_len, _ = context.shape
        context = context.transpose(1, 2).contiguous().view(
            batch_size, query_len, self.n_heads * self.d_k
        )
        output = self.out_proj(context)
        return (output, attention) if return_attention else output

In [44]:
# 演示：“我爱你” -> Token ID [2, 213, 534] -> Encoder 的 X -> Multi-Head Attention
# 因为最大 Token ID 是 534，所以 vocab_size 至少需要为 535；这里取 1000。
DEMO_VOCAB_SIZE = 1000
DEMO_TOKEN_IDS = torch.tensor([[2, 213, 534]], dtype=torch.long, device=DEVICE)

# (vocab_size, d_model)：1000 个 token，每个 token 用 64 维向量表示
demo_embedding = nn.Embedding(DEMO_VOCAB_SIZE, D_MODEL).to(DEVICE)
demo_position = PositionalEncoding(D_MODEL, max_len=MAX_LEN, dropout=0.0).to(DEVICE)
demo_attention = MultiHeadAttention(D_MODEL, N_HEADS, dropout=0.0).to(DEVICE)

with torch.no_grad():
    # Token ID: (1, 3) -> Token Embedding: (1, 3, 64)
    demo_token_vectors = demo_embedding(DEMO_TOKEN_IDS) * math.sqrt(D_MODEL)

    # Position Encoding: (1, 3, 64)，与 Token Embedding 逐元素相加
    demo_pe = demo_position.pe[:, :DEMO_TOKEN_IDS.size(1)]
    demo_x = demo_position(demo_token_vectors)

    # 为了展示形状，单独计算拆分多头后的 Q、K、V
    demo_q = demo_attention._split_heads(demo_attention.q_proj(demo_x))
    demo_k = demo_attention._split_heads(demo_attention.k_proj(demo_x))
    demo_v = demo_attention._split_heads(demo_attention.v_proj(demo_x))
    demo_output, demo_weights = demo_attention(
        demo_x, demo_x, demo_x, return_attention=True
    )

print("文字:                         我 爱 你")
print("Token IDs:                   ", DEMO_TOKEN_IDS.tolist())
print("Token IDs shape:             ", tuple(DEMO_TOKEN_IDS.shape))
print("Embedding 表 shape:          ", tuple(demo_embedding.weight.shape))
print("Token Embedding shape:       ", tuple(demo_token_vectors.shape))
print("Position Encoding shape:     ", tuple(demo_pe.shape))
print("相加后的 X shape:             ", tuple(demo_x.shape))
print("拆分多头后的 Q/K/V shape:     ", tuple(demo_q.shape))
print("Attention 权重 shape:         ", tuple(demo_weights.shape))
print("Multi-Head Attention 输出:    ", tuple(demo_output.shape))
print("X 中‘我’的前 8 个特征:\n", demo_x[0, 0, :8].cpu())

文字:                         我 爱 你
Token IDs:                    [[2, 213, 534]]
Token IDs shape:              (1, 3)
Embedding 表 shape:           (1000, 64)
Token Embedding shape:        (1, 3, 64)
Position Encoding shape:      (1, 3, 64)
相加后的 X shape:              (1, 3, 64)
拆分多头后的 Q/K/V shape:      (1, 4, 3, 16)
Attention 权重 shape:          (1, 4, 3, 3)
Multi-Head Attention 输出:     (1, 3, 64)
X 中‘我’的前 8 个特征:
 tensor([ 15.4493,   9.0949, -11.4913,  -8.0389,  -1.0883,  14.0833,   5.2379,
          5.6080])


In [45]:
class PositionwiseFeedForward(nn.Module):
    def __init__(self, d_model, d_ff, dropout=0.1):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(d_model, d_ff),
            nn.ReLU(),  # 原论文使用 ReLU
            nn.Dropout(dropout),
            nn.Linear(d_ff, d_model),
        )

    def forward(self, x):
        return self.net(x)


class EncoderLayer(nn.Module):
    """原论文 Post-Norm：LayerNorm(x + Sublayer(x))。"""

    def __init__(self, d_model, n_heads, d_ff, dropout=0.1):
        super().__init__()
        self.self_attention = MultiHeadAttention(d_model, n_heads, dropout)
        self.ffn = PositionwiseFeedForward(d_model, d_ff, dropout)
        self.norm1 = nn.LayerNorm(d_model)
        self.norm2 = nn.LayerNorm(d_model)
        self.dropout1 = nn.Dropout(dropout)
        self.dropout2 = nn.Dropout(dropout)

    def forward(self, x, src_mask=None):
        x = self.norm1(x + self.dropout1(self.self_attention(x, x, x, src_mask)))
        x = self.norm2(x + self.dropout2(self.ffn(x)))
        return x


class DecoderLayer(nn.Module):
    def __init__(self, d_model, n_heads, d_ff, dropout=0.1):
        super().__init__()
        self.self_attention = MultiHeadAttention(d_model, n_heads, dropout)
        self.cross_attention = MultiHeadAttention(d_model, n_heads, dropout)
        self.ffn = PositionwiseFeedForward(d_model, d_ff, dropout)
        self.norm1 = nn.LayerNorm(d_model)
        self.norm2 = nn.LayerNorm(d_model)
        self.norm3 = nn.LayerNorm(d_model)
        self.dropout1 = nn.Dropout(dropout)
        self.dropout2 = nn.Dropout(dropout)
        self.dropout3 = nn.Dropout(dropout)

    def forward(self, x, memory, tgt_mask=None, src_mask=None):
        # Masked self-attention：当前位置不能看见未来 token
        x = self.norm1(x + self.dropout1(self.self_attention(x, x, x, tgt_mask)))
        # Cross-attention：Q 来自 Decoder，K/V 来自 Encoder
        x = self.norm2(
            x + self.dropout2(self.cross_attention(x, memory, memory, src_mask))
        )
        x = self.norm3(x + self.dropout3(self.ffn(x)))
        return x


class Encoder(nn.Module):
    def __init__(self, n_layers, d_model, n_heads, d_ff, dropout=0.1):
        super().__init__()
        self.layers = nn.ModuleList([
            EncoderLayer(d_model, n_heads, d_ff, dropout)
            for _ in range(n_layers)
        ])

    def forward(self, x, src_mask=None):
        for layer in self.layers:
            x = layer(x, src_mask)
        return x


class Decoder(nn.Module):
    def __init__(self, n_layers, d_model, n_heads, d_ff, dropout=0.1):
        super().__init__()
        self.layers = nn.ModuleList([
            DecoderLayer(d_model, n_heads, d_ff, dropout)
            for _ in range(n_layers)
        ])

    def forward(self, x, memory, tgt_mask=None, src_mask=None):
        for layer in self.layers:
            x = layer(x, memory, tgt_mask, src_mask)
        return x

In [46]:
class Transformer(nn.Module):
    def __init__(
        self,
        src_vocab_size,
        tgt_vocab_size,
        d_model=512,
        n_heads=8,
        n_layers=6,
        d_ff=2048,
        dropout=0.1,
        max_len=5000,
    ):
        super().__init__()
        self.d_model = d_model
        self.src_embedding = nn.Embedding(src_vocab_size, d_model, padding_idx=PAD_IDX)
        self.tgt_embedding = nn.Embedding(tgt_vocab_size, d_model, padding_idx=PAD_IDX)
        self.position = PositionalEncoding(d_model, max_len, dropout)
        self.encoder = Encoder(n_layers, d_model, n_heads, d_ff, dropout)
        self.decoder = Decoder(n_layers, d_model, n_heads, d_ff, dropout)
        self.generator = nn.Linear(d_model, tgt_vocab_size)
        self._reset_parameters()

    def _reset_parameters(self):
        for parameter in self.parameters():
            if parameter.dim() > 1:
                nn.init.xavier_uniform_(parameter)

    @staticmethod
    def make_src_mask(src):
        # 屏蔽 Encoder 及 Cross-Attention 中的 PAD key
        return src.ne(PAD_IDX).unsqueeze(1).unsqueeze(2)  # (B, 1, 1, S)

    @staticmethod
    def make_tgt_mask(tgt):
        tgt_len = tgt.size(1)
        padding_mask = tgt.ne(PAD_IDX).unsqueeze(1).unsqueeze(2)
        causal_mask = torch.tril(
            torch.ones(tgt_len, tgt_len, dtype=torch.bool, device=tgt.device)
        ).unsqueeze(0).unsqueeze(0)
        return padding_mask & causal_mask  # (B, 1, T, T)

    def encode(self, src, src_mask):
        x = self.src_embedding(src) * math.sqrt(self.d_model)
        return self.encoder(self.position(x), src_mask)

    def decode(self, tgt, memory, tgt_mask, src_mask):
        x = self.tgt_embedding(tgt) * math.sqrt(self.d_model)
        return self.decoder(self.position(x), memory, tgt_mask, src_mask)

    def forward(self, src, tgt_input):
        src_mask = self.make_src_mask(src)
        tgt_mask = self.make_tgt_mask(tgt_input)
        memory = self.encode(src, src_mask)
        decoder_output = self.decode(tgt_input, memory, tgt_mask, src_mask)
        return self.generator(decoder_output)  # (B, T, tgt_vocab_size)

    @torch.no_grad()
    def greedy_decode(self, src, max_new_tokens):
        """从 BOS 开始逐 token 贪心解码。"""
        self.eval()
        src_mask = self.make_src_mask(src)
        memory = self.encode(src, src_mask)
        output = torch.full(
            (src.size(0), 1), BOS_IDX, dtype=torch.long, device=src.device
        )

        for _ in range(max_new_tokens):
            tgt_mask = self.make_tgt_mask(output)
            hidden = self.decode(output, memory, tgt_mask, src_mask)
            next_token = self.generator(hidden[:, -1]).argmax(dim=-1, keepdim=True)
            output = torch.cat([output, next_token], dim=1)
            if torch.all(next_token.eq(EOS_IDX)):
                break
        return output


model = Transformer(
    src_vocab_size=VOCAB_SIZE,
    tgt_vocab_size=VOCAB_SIZE,
    d_model=D_MODEL,
    n_heads=N_HEADS,
    n_layers=N,
    d_ff=D_FF,
    dropout=DROPOUT,
    max_len=MAX_LEN,
).to(DEVICE)

parameter_count = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f"可训练参数量: {parameter_count:,}")

可训练参数量: 239,648


## 训练示例：序列反转

下面构造一个小型 Seq2Seq 任务。例如输入 `[8, 4, 6]`，目标是输出 `[6, 4, 8, EOS]`。

训练时使用 **Teacher Forcing**：

- Decoder 输入：`[BOS, 6, 4, 8]`
- 监督标签：`[6, 4, 8, EOS]`

不同长度的序列使用 `PAD` 补齐，损失函数会忽略这些位置。这个任务只用于验证完整 Transformer 能正确训练和自回归解码。

In [47]:
def make_reverse_batch(batch_size, min_len=3, max_len=10):
    lengths = torch.randint(min_len, max_len + 1, (batch_size,))
    src = torch.full((batch_size, max_len), PAD_IDX, dtype=torch.long)
    tgt_input = torch.full((batch_size, max_len + 1), PAD_IDX, dtype=torch.long)
    tgt_label = torch.full((batch_size, max_len + 1), PAD_IDX, dtype=torch.long)

    for i, length in enumerate(lengths.tolist()):
        tokens = torch.randint(3, VOCAB_SIZE, (length,))
        reversed_tokens = tokens.flip(0)
        src[i, :length] = tokens
        tgt_input[i, 0] = BOS_IDX
        tgt_input[i, 1:length + 1] = reversed_tokens
        tgt_label[i, :length] = reversed_tokens
        tgt_label[i, length] = EOS_IDX

    return src.to(DEVICE), tgt_input.to(DEVICE), tgt_label.to(DEVICE)


# 先做一次前向传播，检查各维度和 mask 是否正确
src, tgt_input, tgt_label = make_reverse_batch(batch_size=4)
logits = model(src, tgt_input)
print("src shape:      ", tuple(src.shape))
print("tgt_input shape:", tuple(tgt_input.shape))
print("logits shape:   ", tuple(logits.shape))
assert logits.shape == (*tgt_input.shape, VOCAB_SIZE)


class NoamScheduler:
    """论文学习率：d_model^(-0.5) * min(step^(-0.5), step*warmup^(-1.5))。"""

    def __init__(self, optimizer, d_model, warmup_steps=400):
        self.optimizer = optimizer
        self.d_model = d_model
        self.warmup_steps = warmup_steps
        self.step_num = 0

    def step(self):
        self.step_num += 1
        lr = (self.d_model ** -0.5) * min(
            self.step_num ** -0.5,
            self.step_num * self.warmup_steps ** -1.5,
        )
        for group in self.optimizer.param_groups:
            group["lr"] = lr
        self.optimizer.step()
        return lr

src shape:       (4, 10)
tgt_input shape: (4, 11)
logits shape:    (4, 11, 32)


In [48]:
# 原论文使用 Adam(beta1=0.9, beta2=0.98, eps=1e-9) 和 label smoothing=0.1
optimizer = torch.optim.Adam(
    model.parameters(), lr=0.0, betas=(0.9, 0.98), eps=1e-9
)
scheduler = NoamScheduler(optimizer, D_MODEL, warmup_steps=400)
criterion = nn.CrossEntropyLoss(ignore_index=PAD_IDX, label_smoothing=0.1)

TRAIN_STEPS = 800
BATCH_SIZE = 64
model.train()

for step in range(1, TRAIN_STEPS + 1):
    src, tgt_input, tgt_label = make_reverse_batch(BATCH_SIZE)
    logits = model(src, tgt_input)
    loss = criterion(logits.reshape(-1, VOCAB_SIZE), tgt_label.reshape(-1))

    optimizer.zero_grad()
    loss.backward()
    torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
    lr = scheduler.step()

    if step == 1 or step % 100 == 0:
        print(f"step {step:4d} | loss={loss.item():.4f} | lr={lr:.6f}")

step    1 | loss=3.9917 | lr=0.000016


step  100 | loss=2.5869 | lr=0.001563
step  200 | loss=1.8680 | lr=0.003125
step  300 | loss=1.6529 | lr=0.004687
step  400 | loss=1.3704 | lr=0.006250
step  500 | loss=1.0427 | lr=0.005590
step  600 | loss=0.8862 | lr=0.005103
step  700 | loss=0.8848 | lr=0.004725
step  800 | loss=0.8610 | lr=0.004419


In [49]:
def without_special_tokens(sequence):
    result = []
    for token in sequence:
        if token == EOS_IDX:
            break
        if token not in (PAD_IDX, BOS_IDX):
            result.append(token)
    return result


# 测试模型未在训练步骤中见过的新序列
examples = [
    [5, 9, 12, 7],
    [18, 4, 25, 6, 11, 3],
    [8, 14, 20],
]
max_src_len = max(map(len, examples))
test_src = torch.full((len(examples), max_src_len), PAD_IDX, dtype=torch.long)
for i, sequence in enumerate(examples):
    test_src[i, :len(sequence)] = torch.tensor(sequence)
test_src = test_src.to(DEVICE)

predictions = model.greedy_decode(test_src, max_new_tokens=max_src_len + 1).cpu().tolist()

print("输入 -> 期望输出 -> Transformer 输出")
for source, prediction in zip(examples, predictions):
    print(f"{source} -> {source[::-1]} -> {without_special_tokens(prediction)}")

输入 -> 期望输出 -> Transformer 输出
[5, 9, 12, 7] -> [7, 12, 9, 5] -> [7, 12, 9, 5]
[18, 4, 25, 6, 11, 3] -> [3, 11, 6, 25, 4, 18] -> [3, 11, 6, 25, 4, 18]
[8, 14, 20] -> [20, 14, 8] -> [20, 14, 8]


In [50]:
input=torch.randint(low=0,high=100,size=(1,3))
print(input)

tensor([[30, 95, 20]])


In [51]:
embedding_table = torch.randn(1000, 64)
print(embedding_table.shape)

torch.Size([1000, 64])


In [52]:
token_embedding = embedding_table[input]
print(token_embedding.shape)  # 应输出: torch.Size([1, 3, 64])

torch.Size([1, 3, 64])


In [54]:
torch.arange(0, 8, 2, dtype=torch.float32)

tensor([0., 2., 4., 6.])